Ingestao Streaming - Camada Bronze (Azure)

Baixa as tabelas do dataset br_inep_avaliacao_alfabetizacao da Base dos
Dados (fonte publica de microdados educacionais) e grava em Parquet sem transformacoes.

**INEP Alunos**
| Nome | Descrição | Tipo (BigQuery) |
| --- | --- | --- |
| ano | Ano de aplicação da avaliação estadual | INT64 |
| id_municipio | ID município de 7 dígitos | STRING |
| id_escola | Máscara do código da escola (fictício) | STRING |
| id_aluno | Código do aluno | STRING |
| caderno | Código do caderno atribuído ao aluno na prova de LP | STRING |
| serie | Ano escolar | STRING |
| rede | Dependência administrativa da escola | STRING |
| presenca | Indicador de presença na prova de LP | STRING |
| preenchimento_caderno | Indicador de preenchimento da prova de LP | STRING |
| alfabetizado | Indica se o aluno é considerado alfabetizado | STRING |
| proficiencia | Proficiência do aluno em LP (escala SAEB) | FLOAT64 |
| peso_aluno | Peso do aluno na prova de LP | FLOAT64 |

---

In [0]:
import io
import random
from datetime import datetime, UTC

import pandas as pd


# ==========================================
# Municípios reais (amostra ampliada)
# ==========================================

MUNICIPIOS = [
    "1100015", "1100023", "1100031", "1100049",
    "1200013", "1200054", "1300029", "1302603",
    "1500107", "1501402", "1600105", "1702109",
    "2100055", "2207702", "2304400", "2408102",
    "2507507", "2607901", "2704302", "2800308",
    "2905701", "2910800", "2927408", "3101706",
    "3118601", "3136702", "3154606", "3169901",
    "3205309", "3304557", "3509502", "3529401",
    "4106902", "4202008", "4216602", "4305108",
    "4314902", "5002704", "5103403", "5104807",
    "5208707", "5300108"
]

ID_ALUNOS = [
    "13011245","15046043","23003926","23041252",
    "23066095","24008090","25003465","25004156",
    "25008575","26000941","26001621","26016071",
    "26016625","26022154","27001760","28002871",
    "29042391","29166363","29905845","31067894",
    "31093203","31114527","35000444","35005548",
    "35016345","35029878","35062762","35144498",
    "35168862","35177061","35210038","35263961",
    "35280318","35343836","35374091","35418437",
    "35423566","41067171","41110534","41111796",
    "41121799","42035513","42064101","43022747",
    "43103500","50033678","51022857","51039254",
    "52029385","52062729","53001602","53008920"
]

ID_ESCOLAS = [
    "60001606","60002587","60002661","60003671",
    "60004015","60006237","60007509","60008388",
    "60008451","60008944","60009346","60010619",
    "60011743","60012162","60012192","60012254",
    "60013273","60013395","60016000","60017246",
    "60017336","60017459","60017571","60018299",
    "60019936","60020036","60020439","60020666",
    "60022309","60023344","60024301","60024667",
    "60025570","60025587","60025734","60026416",
    "60026496","60026936","60027176","60027600",
    "60029337","60030343","60030408","60031087",
    "60032945","60033257","60033416","60034608",
    "60036652","60036687","60037081","60038505",
    "60038783","60039049","60040834","60040927",
    "60041337","60041562","60041587","60041915"
]

# ==========================================
# Domínios reais
# ==========================================

CADERNOS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

SERIES = [2]

REDES = [2, 3, 4]

PRESENCA = [0, 1]

PREENCHIMENTO_CADERNO = [0, 1]

ALFABETIZADO = [0, 1]


# ==========================================
# Registro sintético
# ==========================================

def gerar_registro():

    return {

        "ano": 2025,

        "id_municipio": random.choice(
            MUNICIPIOS
        ),

        "id_escola": random.choice(
            ID_ESCOLAS
        ),

        "id_aluno":random.choice(
            ID_ALUNOS
        ),

        "caderno": str(
            random.choice(
                CADERNOS
            )
        ),

        "serie": str(
            random.choice(
                SERIES
            )
        ),

        "rede": str(
            random.choice(
                REDES
            )
        ),

        "presenca": str(
            random.choice(
                PRESENCA
            )
        ),

        "preenchimento_caderno": str(
            random.choice(
                PREENCHIMENTO_CADERNO
            )
        ),

        "alfabetizado": str(
            random.choice(
                ALFABETIZADO
            )
        ),

        "proficiencia": round(
            random.uniform(
                578.46,
                904.38
            ),
            6
        ),

        "peso_aluno": round(
            random.uniform(
                0.5,
                2.0
            ),
            6
        ),

        "_ingested_at": datetime.now(
            UTC
        ).isoformat(),

        "_source_table": (
            "basedosdados."
            "br_inep_avaliacao_alfabetizacao."
            "alunos"
        )
    }


# ==========================================
# Lote
# ==========================================

def gerar_lote(qtd_registros):

    registros = [
        gerar_registro()
        for _ in range(qtd_registros)
    ]

    return pd.DataFrame(
        registros
    )


# ==========================================
# Upload Parquet
# ==========================================

def upload_parquet(
    blob_service_client,
    container_name,
    dataframe,
    blob_name
):

    parquet_buffer = io.BytesIO()

    dataframe.to_parquet(
        parquet_buffer,
        index=False,
        engine="pyarrow"
    )

    blob_client = (
        blob_service_client.get_blob_client(
            container=container_name,
            blob=blob_name
        )
    )

    blob_client.upload_blob(
        parquet_buffer.getvalue(),
        overwrite=True
    )

    return len(
        dataframe
    )


# ==========================================
# Execução Batch
# ==========================================

def execute_batch(
    blob_service_client,
    container_name,
    records,
    folder
):

    df = gerar_lote(
        records
    )

    data_arquivo = datetime.now(
        UTC
    ).strftime(
        "%Y-%m-%d_%H%M%S"
    )

    blob_name = (
        f"{folder}/"
        f"inep_alunos_"
        f"{data_arquivo}.parquet"
    )

    total_sent = upload_parquet(
        blob_service_client,
        container_name,
        df,
        blob_name
    )

    return {
        "records_sent": total_sent,
        "blob_name": blob_name,
        "preview": df.head()
    }